In [18]:
# 0) Mount Google Drive (VIDVRD zip + outputs)
from google.colab import drive
drive.mount('/content/drive')


/content/STTran
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 9.52 KiB | 3.17 MiB/s, done.
From https://github.com/TommasoAiello08/STTran
   9d8359c..8c52457  main       -> origin/main
Updating 9d8359c..8c52457
Fast-forward
 ...aking_the_Most_of_your_Colab_Subscription.ipynb | 496 +++++++++++++++++++++
 fasterRCNN/lib/model/roi_layers/roi_pool.py        |  33 +-
 2 files changed, 520 insertions(+), 9 deletions(-)
 create mode 100644 Copy_of_Making_the_Most_of_your_Colab_Subscription.ipynb


In [1]:
# 1) Clone repo (edit URL if you use a fork)
%cd /content
!rm -rf STTran
!git clone <YOUR_GITHUB_REPO_URL> STTran
%cd /content/STTran



Mounted at /content/drive


In [4]:
# 2) Setup (idempotent; reruns should be fast)
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u setup_colab.py --colab



/content
Cloning into 'STTran'...
remote: Enumerating objects: 799, done.
remote: Counting objects: 100% (799/799), done.
remote: Compressing objects: 100% (543/543), done.
remote: Total 799 (delta 293), reused 732 (delta 226), pack-reused 0 (from 0)
Receiving objects: 100% (799/799), 10.40 MiB | 1.65 MiB/s, done.
Resolving deltas: 100% (293/293), done.
/content/STTran


In [5]:
# 3) Unzip VIDVRD zip from Drive → local /content (fast)
%%bash
set -euo pipefail
set -x

ZIP="/content/drive/MyDrive/vidvrd-dataset_480.zip"  # <-- adjust if your zip name/path differs
OUT="/content/vidvrd-dataset_480"

# If already unzipped in this runtime, skip
if [ -d "$OUT/test_frames_480" ] && [ -d "$OUT/test_480" ] && \
   [ -d "$OUT/train_frames_480" ] && [ -d "$OUT/train_480" ]; then
  echo "[skip] already unzipped at $OUT"
  exit 0
fi

rm -rf "$OUT" /content/VIDVRD-DATASET_480 /content/VIDVRD
unzip -q "$ZIP" -d /content

# Auto-detect extracted dataset root and normalize to /content/vidvrd-dataset_480
LOCAL_ROOT=""
for CAND in \
  /content/vidvrd-dataset_480 \
  /content/VIDVRD-DATASET_480 \
  /content/VIDVRD-DATASET_480/VIDVRD-DATASET_480 \
  /content/VIDVRD/VIDVRD-DATASET_480 \
  ; do
  if [ -d "$CAND/train_frames_480" ] && [ -d "$CAND/train_480" ]; then
    LOCAL_ROOT="$CAND"
    break
  fi
done

if [ -z "$LOCAL_ROOT" ]; then
  echo "[debug] /content listing:" >&2
  ls -lah /content | head -n 200 >&2
  echo "[error] Could not find extracted VIDVRD dataset root. Check ZIP structure." >&2
  exit 2
fi

# Ensure the canonical path exists
if [ "$LOCAL_ROOT" != "$OUT" ]; then
  rm -rf "$OUT"
  mv "$LOCAL_ROOT" "$OUT"
fi

echo "[ok] dataset_root=$OUT"
ls -lah "$OUT" | head -n 50



[colab] Skipping legacy native extension build (use --compile-native or STTRAN_COMPILE_NATIVE=1 to force).
STTran Colab setup  repo_root=/content/STTran
+ /usr/bin/python3 -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 52.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
[colab] keeping existing torch 2.10.0+cu128 (cuda=True)
+ /usr/bin/python3 -m pip install numpy scipy imageio pillow tqdm six cython ninja matplotlib networkx gdown pyyaml opencv-python pandas dill easydict h5py
[download] http://nlp.stanford.edu/data/glove.6B.zip -> /content/STTran/data/glove.6B.zip
+ /usr/bin/python3 /content/STTran/scripts/download_sttran_ag_weights.py --out_dir /content/STTran/.sttran_weight_cache --link_into_repo
[download] /content/STTran/.sttran_weight_cache/faster_rcnn_ag.pth
[download] /content/STTran/.sttran_weight_cache/object_bbox_and_relations

+ cd /content/STTran
+ python -u setup_colab.py --colab
Downloading...
From (original): https://drive.google.com/uc?id=1-u930Pk0JYz3ivS6V_HNTM1D5AxmN5Bs
From (redirected): https://drive.google.com/uc?id=1-u930Pk0JYz3ivS6V_HNTM1D5AxmN5Bs&confirm=t&uuid=6637b134-b2c9-44f3-8bb0-95923f2329fd
To: /content/STTran/.sttran_weight_cache/faster_rcnn_ag.pth
100%|██████████| 380M/380M [00:02<00:00, 176MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=19BkAwjCw5ByyGyZjFo174Oc3Ud56fkaT
From (redirected): https://drive.google.com/uc?id=19BkAwjCw5ByyGyZjFo174Oc3Ud56fkaT&confirm=t&uuid=3b97114c-314f-4597-90cc-96c4658657b0
To: /content/STTran/.sttran_weight_cache/object_bbox_and_relationship_filtersmall.pkl
100%|██████████| 136M/136M [00:00<00:00, 228MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Sk5qFLWTZmwr63fHpy_C7oIxZSQU16vU
From (redirected): https://drive.google.com/uc?id=1Sk5qFLWTZmwr63fHpy_C7oIxZSQU16vU&confirm=t&uuid=eebd4634-da16-4126-9011-a81f52320607

In [6]:
# 4) Optional: if you keep big weights on Drive instead of inside the repo, set these.
# Otherwise, skip this cell.
import os

# os.environ["FASTER_RCNN_AG_PTH"] = "/content/drive/MyDrive/weights/faster_rcnn_ag.pth"
# os.environ["STTRAN_CKPT"] = "/content/drive/MyDrive/weights/sttran_predcls.tar"



In [13]:
# 5) Dataset layout sanity check (local)
%%bash
set -euo pipefail
set -x

ROOT="/content/vidvrd-dataset_480"
ls -lah "$ROOT" | head -n 50
ls -lah "$ROOT/train_480" | head -n 20
ls -lah "$ROOT/train_frames_480" | head -n 20


Process is terminated.


In [14]:
# 6) Build + save stable vocab (objects + predicates) for real training
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/vidvrd-dataset_480" \
  --vocab_scan_dir "/content/vidvrd-dataset_480/train_480" \
  --save_vocab_json "/content/vidvrd_vocab.json" \
  --video_id "ILSVRC2015_train_00008004" \
  --expected_hw "480,854" \
  --max_frames 8 \
  --no_forward \
  --num_predicates 0

ls -lah /content/vidvrd_vocab.json


[skip] dataset already present on Drive: /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480


+ ZIP=/content/drive/MyDrive/vidvrd-dataset_480.zip
+ DRIVE_ROOT=/content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480
+ '[' -d /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480/test_frames_480 ']'
+ '[' -d /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480/test_480 ']'
+ '[' -d /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480/train_frames_480 ']'
+ '[' -d /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480/train_480 ']'
+ echo '[skip] dataset already present on Drive: /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480'
+ exit 0


In [15]:
# 7) Quick sanity check: load vocab and show sizes
import json
from pathlib import Path

v = json.loads(Path('/content/vidvrd_vocab.json').read_text())
print('num objects:', len(v['object_categories']))
print('num predicates (no background):', len(v['predicate_names']))
print('head size with background:', 1 + len(v['predicate_names']))



DATASET_ROOT: /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480
has test_frames_480: True
has test_480: True
has train_frames_480: True
has train_480: True


In [26]:
# 8) Pipeline test (real vocab) — build tensors only (fast)
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/vidvrd-dataset_480" \
  --vocab_json "/content/vidvrd_vocab.json" \
  --video_id "ILSVRC2015_train_00008004" \
  --expected_hw "480,854" \
  --max_frames 32 \
  --no_forward \
  --num_predicates 0


loading word vectors from /content/STTran/data/glove.6B.200d.pt
__background__ -> __background__ 
fail on __background__
ok: True
WARN: len(trajectories)=210 != frame_count=219 in JSON; pipeline will min-truncate to im_data length.
WARN: object_categories / predicate_names not provided; built vocab from this JSON only. For real training pass the full official lists so indices are stable.
diagnostics: {
  "video_id": "ILSVRC2015_train_00010001",
  "json_frame_count": 219,
  "json_wh": [
    854,
    480
  ],
  "num_trajectory_frames": 210,
  "num_relations": 33,
  "T_use": 32,
  "T_disk": 219,
  "num_vidvrd_predicates_inferred": 14,
  "im_data_shape": [
    32,
    3,
    600,
    1068
  ],
  "im_info_first_row": [
    600.0,
    1068.0,
    1.25
  ],
  "prep_scales_first8": [
    1.25,
    1.25,
    1.25,
    1.25,
    1.25,
    1.25,
    1.25,
    1.25
  ],
  "skipped_relation_msgs": 0,
  "vidvrd_logits_shape": [
    736,
    14
  ]
}
entry keys: ['boxes', 'features', 'im_idx', 'label

+ cd /content/STTran
+ python -u lib/vidvrd_pipeline_validate.py --dataset_root /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480 --video_id ILSVRC2015_train_00010001 --expected_hw 480,854 --max_frames 32 --num_predicates 0
loading word vectors from /content/STTran/data/glove.6B.200d.txt: 100%|██████████| 400000/400000 [00:20<00:00, 19495.84it/s]


In [27]:
# 9) Pipeline test (real vocab) — full forward pass (loads weights, runs model)
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/vidvrd-dataset_480" \
  --vocab_json "/content/vidvrd_vocab.json" \
  --video_id "ILSVRC2015_train_00008004" \
  --expected_hw "480,854" \
  --max_frames 32 \
  --num_predicates 0


loading word vectors from /content/STTran/data/glove.6B.200d.pt
loading word vectors from /content/STTran/data/glove.6B.200d.pt
__background__ -> __background__ 
fail on __background__
ok: True
WARN: len(trajectories)=210 != frame_count=219 in JSON; pipeline will min-truncate to im_data length.
WARN: object_categories / predicate_names not provided; built vocab from this JSON only. For real training pass the full official lists so indices are stable.
diagnostics: {
  "video_id": "ILSVRC2015_train_00010001",
  "json_frame_count": 219,
  "json_wh": [
    854,
    480
  ],
  "num_trajectory_frames": 210,
  "num_relations": 33,
  "T_use": 32,
  "T_disk": 219,
  "num_vidvrd_predicates_inferred": 14,
  "im_data_shape": [
    32,
    3,
    600,
    1068
  ],
  "im_info_first_row": [
    600.0,
    1068.0,
    1.25
  ],
  "prep_scales_first8": [
    1.25,
    1.25,
    1.25,
    1.25,
    1.25,
    1.25,
    1.25,
    1.25
  ],
  "skipped_relation_msgs": 0,
  "vidvrd_logits_shape": [
    736,

+ cd /content/STTran
+ python -u lib/vidvrd_pipeline_validate.py --dataset_root /content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480 --video_id ILSVRC2015_train_00010001 --expected_hw 480,854 --max_frames 32 --num_predicates 0


In [28]:
# 10) Training stack smoke test (synthetic data) — checks optimizer/loss/checkpointing
%cd /content/STTran
!python colab/vidvrd_train_colab.py \
  --out_dir "/content/drive/MyDrive/vidvrd_runs/smoke_synth" \
  --epochs 1 \
  --synthetic


/content/STTran
loading word vectors from /content/STTran/data/glove.6B.200d.pt
loading word vectors from /content/STTran/data/glove.6B.200d.pt
__background__ -> __background__ 
fail on __background__
device=cuda:0  trainable params=255684
epoch 1/1  synthetic_loss=4.8810
  saved /content/drive/MyDrive/vidvrd_runs/smoke_synth/checkpoints/epoch_001.pt (full)
Done.


In [29]:
# (Optional) Quick check: print GPU availability
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))


torch 2.10.0+cu128
cuda available True
gpu NVIDIA A100-SXM4-80GB


In [30]:
# NOTE
# - We unzip the VIDVRD zip to local /content each new runtime (fast).
# - For persistence across sessions, you *can* copy to Drive, but it is often slow/hangs.
pass



In [31]:
# Placeholder cell (kept to minimize diffs). You can ignore it.
pass


In [32]:
# Placeholder cell (kept to minimize diffs). You can ignore it.
pass


In [41]:
# (Optional) Python-only validation entrypoint.
# Prefer the CLI cells above (they match what you'll run in a fresh session).
pass


num objects: 34
num predicates (no background): 130
head size with background: 131
ok: False
errors: ['build_vidvrd_predcls_entry failed: too many indices for tensor of dimension 1']
warnings: ['len(trajectories)=555 != frame_count=560 in JSON; pipeline will min-truncate to im_data length.']
vidvrd_logits_shape: None


In [51]:
# (Optional) Debug helper: include tracebacks on failure (real vocab)
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/vidvrd-dataset_480" \
  --vocab_json "/content/vidvrd_vocab.json" \
  --video_id "ILSVRC2015_train_00008004" \
  --expected_hw "480,854" \
  --max_frames 32 \
  --no_forward \
  --num_predicates 0 \
  --debug_trace


ok: False
ERROR: num_vidvrd_predicates=131 != len(pred2id)=14 (with reserve_background_id0, len(pred2id)=1+|predicate_names|)
WARN: len(trajectories)=210 != frame_count=219 in JSON; pipeline will min-truncate to im_data length.
WARN: object_categories / predicate_names not provided; built vocab from this JSON only. For real training pass the full official lists so indices are stable.
diagnostics: {
  "video_id": "ILSVRC2015_train_00010001",
  "json_frame_count": 219,
  "json_wh": [
    854,
    480
  ],
  "num_trajectory_frames": 210,
  "num_relations": 33,
  "T_use": 32,
  "T_disk": 219
}


+ cd /content/STTran
+ python -u lib/vidvrd_pipeline_validate.py --dataset_root /content/vidvrd-dataset_480 --video_id ILSVRC2015_train_00010001 --expected_hw 480,854 --max_frames 32 --num_predicates 131 --debug_trace


In [37]:
# (Optional) Old Drive-copy helper removed (we avoid Drive sync by default).
pass


Process is interrupted.


In [ ]:
# End.
